# Connaissance et pratique de la Science Ouverte

In [ ]:
import pandas as pd

import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket
import scipy.stats

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [ ]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-07-24.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep =",")

list_affil = pd.read_csv("list_affiliation.csv", sep =",")
df0 = df0.loc[~df0.q45_clé.isin(["9EAP-NB4B","QNYZ-3MH2"])].merge(list_affil, on = ["q45_clé", "q44_ufr_labo"], how = "left")

In [ ]:

df_col = pd.read_csv("../le_questionnaire/dico_variable.csv", sep = ",")


In [ ]:
list_nominal_simple = [x for x in df_col.label.loc[(df_col.type.isin(["simple_nominal", "booléen","ordinal"]))]]


In [ ]:
def split_multiple_choices(data, column, index, sep = '|'):
    """
    split and explode column with multiple value

    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_split = data.copy()
    df_split[column]= df_split.apply(lambda row: row[column].replace(";","|") ,1 )
    df_split[column] = df_split[column].str.split(sep)
    df_explode = df_split.explode(column)
    gb_data = df_explode.groupby([column]).agg(nb = (index, "size")).sort_values("nb", ascending=False).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100
    
    return df_explode, gb_data
    

In [ ]:
def grouped_question(data, column, index = "q45_clé"):
    """


    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_tmp = data.copy()
    gb_data = df_tmp.groupby([column]).agg(nb = (index, "size")).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100

    return gb_data

In [ ]:
df0

In [ ]:
df_col["question_family"] = df_col.label.apply(lambda row : row.split("_")[0])

dict_question = dict(zip(df_col.label.loc[df_col.label.isin(list_nominal_simple)], df_col.question_family.loc[df_col.label.isin(list_nominal_simple)]))


Les premières questions interrogeaient les participants sur leurs connaissances et leurs pratiques de la Science Ouverte, thème central de l'enquête. Le questionnaire commencait ainsi par demander aux participants de préciser leur degré de familiarité avec les principes de la Science Ouverte. 56% ont répond "oui, un peu", 32 "oui, tout à fait" et 12% "non". Les pratiques étaient ensuite abordaient à travers des questions sur le dépôt des publications sur Hal et Octaviana, et l'ouverture de données de recherche.

40 % des personnes interrogées ont déposé plusieurs fois leurs publications sur Hal et 35% le font systématiquement. 13%  ne l'ont jamais fais. Concernant Octaviana, un peu moins de 10% des répondants ont déjà déposé des productions sur la bibliothèque numérique. Ces productions sont des captations d'événements pour trois d'entre elles et des travaux universitaires et des publications pour les huit autres. En tout une personne interrogée sur cinq n'a jamais entendu parlé d'Octaviana.

| Principes de la science ouverte   |   nb |    freq |
|:-------------------|-----:|--------:|
| Non                |   14 | 11,8 |
| Oui, tout à fait   |   38 | 31,9 |
| Oui, un peu        |   67 | 56,3 |
|Total               |119   | 100,0|


|Dépôt dans Hal        |   nb |    freq |
|:----------------------|-----:|--------:|
| Non                   |   15 | 12,6  |
| Oui, plusieurs fois   |   48 | 40,3 |
| Oui, systématiquement |   41 | 34,5 |
| Oui, une fois         |   15 | 12,6 |
|Total               |119   | 100,0|


| Dépot sur Octaviana  |   nb |    freq |
|:----------------|-----:|--------:|
| Non             |  108 | 90,8 |
| Oui             |   11 |  9,2|
|Total               |119   | 100,0|

| Ouverture des données   |   nb |   total |      freq |  
|:------------------------|-----:|--------:|----------:|
| Non                     |   86 |     119 | 72,3  |      
| Autre, précisez         |   24 |     119 | 20,7   |          
| Oui, sur Nakala         |    5 |     119 |  4,2  |          
| Oui, sur Zenodo |    4 |     119 |  3,4  |         
| Oui, sur Data Paris 8   |    1 |     119 |  0,8 |        
  


Enfin, 28% des personnes interrogées ont déjà produit des données ouvertes. 8% d'entre elles ont utilisé au moins une fois des entrepôts généralistes comme Nakala (N=5), Zenodo (N=4) ou Data Paris 8 (N=1). Les autres modalité d'ouverture des données sont Open Science Framework (N=10), des sites web personnels (N=6), des dépôts "git" (N=3) et des entrepots spécialisés (N=3) liés à la linguistiques : Ortolang et Cocoon.

![](viz/autre_entrepot_rec.png)



In [ ]:
!pip install tabulate

In [ ]:
so_principe = grouped_question(df0, column="q1_so_principles", index = "q45_clé")
print(so_principe[["q1_so_principles", "nb","freq"]].to_markdown(index=False))
depot_hal = grouped_question(df0, column="q2_hal_depot", index = "q45_clé")
print("\n### Dépôt dans hal\n", depot_hal[["q2_hal_depot", "nb","freq"]].to_markdown(index=False))
octaviana = grouped_question(df0, column="q3_octavi_rec", index = "q45_clé")
print("\n### Dépôt dans Octaviana\n", octaviana[["q3_octavi_rec", "nb","freq"]].to_markdown(index=False))

df_exp, octaviana2 = split_multiple_choices(df0, column="q3_octavi_depot", index="q45_clé", sep = '|')
print("\n### Dépôt dans Octaviana détaillé \n", octaviana2.to_markdown())



In [ ]:
df0.loc[df0.q4_diff_data !="Non", "q4_diff_data_grouped"] = "Oui"
df0.loc[df0.q4_diff_data =="Non", "q4_diff_data_grouped"] = "Non"

depot_donnee = grouped_question(df0, column="q4_diff_data_grouped", index = "q45_clé")
print("\n### Dépôt des données de recherche\n", depot_donnee)

df_exp, depot_donnee2 = split_multiple_choices(df0, column="q4_diff_data", index="q45_clé", sep = '|')
print("\n### Dépôt dans Octaviana détaillé \n", depot_donnee2.to_markdown(index=False))

## Les autres modes de dépôt des données de recherche

In [ ]:
df0.loc[df0["q4_diff_data"]=='Autre, précisez', "q4_diff_data_rec"] = df0.q4_autres_entrepots_rec
df0.loc[df0["q4_diff_data"]!='Autre, précisez', "q4_diff_data_rec"] = df0.q4_diff_data
df0.loc[df0["q4_diff_data_rec"].isna(), "q4_diff_data_rec"] = "NSP"
q4 = split_multiple_choices(df0, column="q4_diff_data_rec", index="q45_clé", sep = '|')

In [ ]:
# Initialize the matplotlib figure
fig, ax = plt.subplots(1, figsize=(6,4))

# Plot the total crashes
sns.set_color_codes("pastel")
q4_other = df0.loc[df0.q4_diff_data=="Autre, précisez"].fillna("Non précisé")
col= "q4_autres_entrepots_rec"
df_exp, gb_data = split_multiple_choices(q4_other, column=col, index="q45_clé", sep = '|')
sns.barplot(x="total", y=col, data=gb_data,
            label="Non", color="b", ax=ax)
sns.barplot(x="nb", y=col, data=gb_data,
            label="Oui", color="r", ax=ax)
titre = "Les autres lieux de dépot des données de recherche"
ax.yaxis.set_label_text("")
ax.xaxis.set_label_text("")
ax.set_title(titre.replace("<i>","(").replace("</i>",")"))
    

sns.despine(left=True, bottom=True)
plt.savefig(f"viz/autre_entrepot_rec.png", bbox_inches='tight', dpi = 200)



## Les besoins d'accompagnement à la Science Ouverte

Mis à part l'accompagnement au dépot dans Hal considéré comme inutile par 68% des répondants, une majorité d'entre eux sont favorables aux accompagnement à la Science Ouverte comme étant utiles. Les licences de diffusion des résultats de recherche est le sujet pour lequel les répondants ont manifesté le plus d'intérêt : 87% considèrent qu'un accompagnement sur ce thème serait "plutôt utile" (52%) voire "très utile" (35%). Ensuite, 81% des participants à l'enquêtre déclarent qu'il serait utile d'être accompagner dans le processus permettant de produire des données respectant les principes FAIR (Facile à trouver, Accessible, Interopérable, et Réutilisable). De façon général, les accompagnements apparaissent d'autant plus utiles qu'ils portent sur des aspects juridiques (licence, RGPD) ou des sujets liés à l'ouverture des données (métadonnées, dépots, principes FAIR).

![](viz/util_accomp_so.png)


In [ ]:
for col in df_col.label.loc[(df_col.group=="6_help_SO")&(df_col.type=="ordinal")]:
    gb_data= grouped_question(df0, column=col, index="q45_clé")
    print(gb_data.to_markdown())

In [ ]:
[col for col in df_col.label.loc[(df_col.group=="6_help_SO")&(df_col.type=="ordinal")]]

In [ ]:
dict_accomp = {'q8_depot_hal_help':"Dépot sur Hal",
 'q8_rediger_pdg_help': 'Rédiger un PGD',
 'q8_respect_rgpd_help': 'Respecter le RGPD',
 'q8_licences_help': 'Choix des licences de diffusion',
 'q8_depot_donnees_help': 'Déposer des données',
 'q8_metadonnees_help': 'Connaître les standards de métadonnées',
 'q8_data_paper_help':'Rédiger un data paper',
 'q8_fair_data_help':'Fairiser les données'
              }

In [ ]:

i=-1
for col in df_col.label.loc[(df_col.group=="6_help_SO")&(df_col.type=="ordinal")]:
    i+=1
    dftmp = df0[["q45_clé", col]].rename(columns={col:"interet_accomp"})
    dftmp["accomp_name"]= dict_accomp[col]


    if i == 0:
        q8 = dftmp.copy()
    else:
        q8 = pd.concat([q8, dftmp])


In [ ]:
q8_dis = pd.crosstab(q8.accomp_name, q8.interet_accomp, normalize="index").cumsum(axis=1).stack().reset_index(name='nb').sort_values(by=["accomp_name","interet_accomp"],
                                                                                                                                     ascending=[True, False])


In [ ]:
fig, ax = plt.subplots(1, figsize=(6,4))

sns.set_theme(style="ticks", context="paper")

   
g = sns.barplot(x="nb", y="accomp_name", data=q8_dis, hue="interet_accomp", dodge=False)

titre = "L'utilité des accompagnements à la science ouverte"
ax.yaxis.set_label_text("")
ax.xaxis.set_label_text("")
ax.set_xticks(ticks = [x/10 for x in range(11)])
ax.set_title(titre.replace("<i>","(").replace("</i>",")"))
ax.legend(bbox_to_anchor=(1.05, 1),
                         loc='upper left', borderaxespad=0.)

sns.despine(left=True, bottom=True)
plt.savefig(f"viz/util_accomp_so.png", bbox_inches='tight', dpi = 200)

In [ ]:
fig, ax = plt.subplots(1, figsize=(6,4))

sns.set_color_codes("pastel")

q8_grp = q8.groupby(["accomp_name","interet_accomp"]).agg(nb=("q45_clé", "size")).sort_values(by=["accomp_name","nb"], ascending=[True, False])

    
sns.barplot(x="nb", y="accomp_name", data=q8_grp, hue="interet_accomp", dodge=True)
titre = "L'utilité des accompagnements à la science ouverte"
ax.yaxis.set_label_text("")
ax.xaxis.set_label_text("")
ax.set_title(titre.replace("<i>","(").replace("</i>",")"))
    

sns.despine(left=True, bottom=True)
#plt.savefig(f"viz/autre_entrepot_rec.png", bbox_inches='tight', dpi = 200)

# Production et analyse des données

## Types de données produites

Les personnes interrogées travaillant exclusivement avec des données dites "quantitatives" sont minoritaires, près de la moitié s'appuient essentiellement sur des données qualitatives et environ un tiers manipulent les deux types de données.

| q11_quali_or_quanti       |   nb |    freq |
|:--------------------------|-----:|--------:|
| Des données qualitatives  |   57 | 47,9 |
| Des données quantitatives |   19 | 16,0 |
| Les deux                  |   43 | 36,1 |
|Total               |119   | 100,0|

Les quatre types de données les plus fréquemment produites sont les entretiens, les données textuelles, les données d'enquête et les données expérimentales ou d'observation. On notera toutefois le caractère ambigü de certaines catégories. Ainsi la notion d'"enquête" est largement utilisé en sciences sociales pour désigner la partie empirique des recherche et peut renvoyer aussi bien aux méthodes d'enquêtes ethnographiques qu'aux méthodes d'enquêtes par questionnaire. De même, les données d'observation peuvent se comprendre au sens de l'anthropologue qui observe les interactions au sein d'un groupe humain  ou de l'ornithologue qui compte les espèces d'oiseau. Parmi les "autres types de données", une personne a d'ailleurs précisé qu'il s'agissait d "observations in situ non expérimentales". Tandis qu'une autre ne se reconnait pas dans la notion de "donnée".



![](viz/type_donnee_quali.png)



In [ ]:
quali_or_quanti = grouped_question(df0, column="q11_quali_or_quanti", index = "q45_clé")
print(quali_or_quanti[["q11_quali_or_quanti","nb","freq"]].to_markdown(index=False))


df_exp, type_data = split_multiple_choices(df0, column="q10_type_data", index="q45_clé", sep = '|')
print("\n### Dépôt dans Octaviana détaillé \n", type_data.to_markdown())

In [ ]:
# Initialize the matplotlib figure
fig, ax = plt.subplots(1, figsize=(6,4))

# Plot the total crashes
sns.set_color_codes("pastel")

col= "q10_type_data"
df_exp, gb_data = split_multiple_choices(df0, column=col, index="q45_clé", sep = '|')
sns.barplot(x="total", y=col, data=gb_data,
            label="Non", color="b", ax=ax)
sns.barplot(x="nb", y=col, data=gb_data,
            label="Oui", color="r", ax=ax)
titre = "Les types de données produites"
ax.yaxis.set_label_text("")
ax.xaxis.set_label_text("")
ax.set_title(titre.replace("<i>","(").replace("</i>",")"))
    

sns.despine(left=True, bottom=True)
plt.savefig(f"viz/type_donnee.png", bbox_inches='tight', dpi = 200)

In [ ]:
def tab_croise(data, x, y, regroup_y = True, khideux = False) :
    """
    x = variable en ligne
    y = variable en colonne
    """
    if regroup_y == True:

        data.loc[data[y].str.lower().str.contains("oui"), f"{y}_rec"] = "Oui"
        data.loc[data[y].str.lower().str.contains("non"), f"{y}_rec"] = "Non"

        cross_tab = pd.crosstab(data[x], data[f"{y}_rec"], margins = True, margins_name="Total", normalize=False)
        cross_tab0 = pd.crosstab(data[x], data[f"{y}_rec"], margins = False)
    else:
        cross_tab = pd.crosstab(data[x], data[y], margins = True, margins_name="Total", normalize=False)
        cross_tab0 = pd.crosstab(data[x], data[y], margins = False, normalize=False)

    if khideux == True:
        print(scipy.stats.chi2_contingency(cross_tab0))
        st_chi2, st_p, st_dof, st_exp = scipy.stats.chi2_contingency(cross_tab0)
        chi2 = pd.DataFrame(data={"stats":["chi2","df","p-value"], "values":[st_chi2, st_dof, st_p]})
        cross_tab = pd.concat([cross_tab, chi2])
    else:
        pass

    return cross_tab.fillna("").reset_index(), cross_tab0

In [ ]:
col= "q10_type_data"
df_exp, gb_data = split_multiple_choices(df0, column=col, index="q45_clé", sep = '|')
df_exp.q10_type_data

In [ ]:
cross_tab, cross_tab0 = tab_croise(df_exp,  "q10_type_data", "q11_quali_or_quanti", regroup_y = False, khideux = False)
q10 = cross_tab0.cumsum(axis=1).stack().reset_index(name='nb').sort_values(by=["nb", "q11_quali_or_quanti"],  ascending=[False, False])
cross_tab.sort_values(by=["Total"],  ascending=[False])

In [ ]:
fig, ax = plt.subplots(1, figsize=(6,4))

sns.set_theme(style="ticks", context="paper")

   
g = sns.barplot(x="nb", y="q10_type_data", data=q10, hue="q11_quali_or_quanti", dodge=False)

titre = "Types de données produites"
ax.yaxis.set_label_text("")
ax.xaxis.set_label_text("")
#ax.set_xticks(ticks = [x/10 for x in range(11)])
ax.set_title(titre.replace("<i>","(").replace("</i>",")"))


sns.despine(left=True, bottom=True)
plt.savefig(f"viz/type_donnee_quali.png", bbox_inches='tight', dpi = 200)

## Autres types de données

In [ ]:
for x in df0.q10_other_type_data.loc[~df0.q10_other_type_data.isna()]:
    print(x)

## Outils de traitement



In [ ]:
i=-1
for x in df_col.label.loc[(df_col.question_family=="q5")& (~df_col.label.str.contains("q5_1"))]:
    i+=1
    gb_data0 = grouped_question(df0, column=x, index = "q45_clé")
    gb_data = gb_data0.loc[gb_data0[x]=="Oui"]
    gb_data.loc[gb_data[x]=="Oui", "logiciel_traitement"] = x.replace("q5_","")
    
    if i == 0:
        gb_q5 = gb_data.drop(columns=[x])#.copy()
    else:
        gb_q5 = pd.concat([gb_q5, gb_data.drop(columns=[x])])
    print(gb_data[[x,"nb","freq"]].to_markdown(index=False))


In [ ]:
# Initialize the matplotlib figure
fig, ax = plt.subplots(1, figsize=(6,4))

# Plot the total crashes
sns.set_color_codes("pastel")

col = "logiciel_traitement"
sns.barplot(x="total_freq", y=col, data=gb_q5.sort_values("nb", ascending=False),
            label="Non", color="b", ax=ax)
sns.barplot(x="nb", y=col, data=gb_q5.sort_values("nb", ascending=False),
            label="freq", color="r", ax=ax)
titre = "Les logiciels de traitement utilisés"
ax.yaxis.set_label_text("")
ax.xaxis.set_label_text("")
ax.set_title(titre.replace("<i>","(").replace("</i>",")"))
    

sns.despine(left=True, bottom=True)
plt.savefig(f"viz/logiciel_traitement.png", bbox_inches='tight', dpi = 200)

In [ ]:
rec_other_lang ={}
for x in df0.q5_1_other_lang.loc[~df0.q5_1_other_lang.isna()]:
    rec_other_lang[x] = x.lower().replace(", ","|")

rec_other_lang

In [ ]:
rec_other_lang = {'MaxQda, EndNotes, Zotero, ': 'maxqda|endnotes|zotero',
 '\u200b-': 'aucun',
 'Matlab': 'matlab',
 'Zotero': 'zotero',
 'LibreOffice Calc': 'libreoffice calc',
 'csv, google sheets': 'csv|google sheets',
 'Jamovi, Iramuteq, Qualcoder': 'jamovi|iramuteq|qualcoder',
 'Nvivo': 'nvivo',
 'php, javascript, d3.js': 'php|javascript|d3.js',
 'Jamovi': 'jamovi',
 'JAMOVI ': 'jamovi',
 'c#': 'c#',
 'SPSS, Statistica, Jamovi, JASP': 'spss|statistica|jamovi|jasp',
 'Nvivo, IA': 'nvivo|ia',
 'Nvivo et Atlas.ti': 'nvivo|atlas.ti',
 'SPSS': 'spss',
 'OCaml, C, Bash, PHP': 'ocaml|c/c++|bash|php',
 'CSS, html, xml': 'css|html|xml',
 'LaTeX': 'latex',
 'NVivo': 'nvivo',
 'MatLab': 'matlab',
 "QGIS. J'aimerais utiliser Python, mais il faudrait que je sois formée pour en avoir les bases. J'ai déjà suivi la formation de base de R proposée à P8, qui était bien : reste maintenant à se l'approprier !": "qgis",
 'Access': 'access',
 'File maker pro': 'file maker pro',
 'LibreOffice Calc, Limesurvey, Grist, ActiveTigger, Neo4j, Gephi': 'libreoffice calc|limesurvey|grist|activetigger|neo4j|gephi',
 'Jamovi, SPSS': 'jamovi|spss',
 'C/C++ Cuda Caml Racket Ludii GDL ASP': 'c/c++|cuda|caml|racket|ludii|gdl|asp',
 'R mais avec Iramutex': 'iramuteq',
 'jamovi': 'jamovi',
 'QSR NVivo': 'nvivo',
 'Mplus, SPSS..': 'mplus|spss'}



df0["q5_1_other_lang_rec"]= df0.q5_1_other_lang.map(rec_other_lang.get)

# on enregistre les nouvelles variables dans le dataframe original
#with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-05-11.csv", "w") as file_out:
#    df0.to_csv(file_out, sep=",", index= False)


In [ ]:
dic_new_var = {'name':'14. other_lang',
               'label':'q5_1_other_lang_rec',
               'group':'4_logiciel_traitement',
               'personal_data':False,
               'type':'multiple_choice',
               'opened_question':False,
               'type_panda':df0.q5_1_other_lang_rec.dtypes,
               'no_question':14,
               'question':df_col.question.loc[df_col.label=="q5_1_other_lang"].iloc[0],
               "question_family":df_col.question_family.loc[df_col.label=="q5_1_other_lang"].iloc[0],
               'comment':"Recodage de la question 14. mise en basse casse des logiciels cités."
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col = pd.concat([df_col,new_variable], ignore_index = True)
df_col

In [ ]:
# Initialize the matplotlib figure
fig, ax = plt.subplots(1, figsize=(6,10))

# Plot the total crashes
sns.set_color_codes("pastel")

q5_1 = df0.loc[~df0.q5_1_other_lang_rec.isna()]
col= "q5_1_other_lang_rec"
df_exp, gb_data = split_multiple_choices(q5_1, column=col, index="q45_clé", sep = '|')
sns.barplot(x="total", y=col, data=gb_data,
            label="Non", color="b", ax=ax)
sns.barplot(x="nb", y=col, data=gb_data,
            label="Oui", color="r", ax=ax)
titre = "Les types de données produites"
ax.yaxis.set_label_text("")
ax.xaxis.set_label_text("")
ax.set_title(titre.replace("<i>","(").replace("</i>",")"))
    

sns.despine(left=True, bottom=True)
#plt.savefig(f"viz/type_donnee.png", bbox_inches='tight', dpi = 200)

In [ ]:
list_other_logi = [x for x in df_exp.q5_1_other_lang_rec.unique()]
print(list_other_logi)

In [ ]:
family_logi = {'maxqda':'CAQDAS', 
               'endnotes':'gestion biblio', 
               'zotero':'gestion biblio', 
               'aucun':'aucun', 
               'matlab':'script et programmation', 
               'libreoffice calc':'feuille de calcul', 
               'csv':'feuille de calcul', 
               'google sheets':'feuille de calcul',
               'jamovi':'logiciels de stats',
               'iramuteq':'analyse de texte', 
               'qualcoder':'CAQDAS',
               'nvivo':'CAQDAS',
               'php':'développement web', 
               'javascript':'développement web',
               'd3.js':'développement web', 
               'c#':'script et programmation', 
               'spss':'logiciels de stats',
               'statistica':'logiciels de stats',
               'jasp':'logiciels de stats',
               'ia':'ia', 
               'atlas.ti':'CAQDAS',
               'ocaml':'script et programmation', 
               'c/c++':'script et programmation',
               'bash':'script et programmation',
               'css':'développement web',
               'html':'développement web', 
               'xml':'développement web',
               'latex':'script et programmation',
               'qgis':'cartographie',
               'access':'gestion de base de donnée',
               'file maker pro':'gestion de base de donnée',
               'limesurvey':'logiciel d\'enquête statistique',
               'grist':'gestion de base de donnée',
               'activetigger':'analyse de texte',
               'neo4j':'gestion de base de donnée',
               'gephi':'analyse de réseau',
               'cuda':'script et programmation', 
               'caml':'script et programmation', 
               'racket':'script et programmation', 
               'ludii':'création de jeu',
               'gdl':'logiciels de stats',
               'asp':'développement web',
               'mplus': 'logiciels de stats',
               'excel': 'feuille de calcul',
               'python': 'script et programmation',
               'r':'script et programmation',
               'stata':'logiciels de stats',
               'julia': 'script et programmation',
               'sas': 'logiciels de stats'
              }

In [ ]:
df01 = df0.copy()

dict_logiciel_traitement = {}
for row in df0.q45_clé:
    logiciel_traitement = []
    for x in df_col.label.loc[(df_col.question_family=="q5")& (~df_col.label.str.contains("q5_1"))]:
        if df01[x].loc[df01.q45_clé==row].values == "Oui":
            logiciel_traitement.append(x.lower().replace("q5_",""))
    if len(logiciel_traitement)>0:
        dict_logiciel_traitement[row]= '|'.join(logiciel_traitement)
df01["q5_logiciel_traitement_rec"] = df01.q45_clé.map(dict_logiciel_traitement)



In [ ]:
df0
df_exp, gb_data = split_multiple_choices(df01.loc[~df01.q5_logiciel_traitement_rec.isna()], column='q5_logiciel_traitement_rec', index="q45_clé", sep = '|')
df_exp1, gb_data1 = split_multiple_choices(df01.loc[~df01.q5_1_other_lang_rec.isna()], column='q5_1_other_lang_rec', index="q45_clé", sep = '|')

df_exp2 = pd.concat([df_exp[["q45_clé", 'q5_logiciel_traitement_rec']], df_exp1[["q45_clé","q5_1_other_lang_rec"]].rename(columns={"q5_1_other_lang_rec":'q5_logiciel_traitement_rec'})])
df_exp2["q5_family_logi"]= df_exp2.q5_logiciel_traitement_rec.map(family_logi.get)
df_exp3 = df_exp2.loc[~df_exp2.q5_logiciel_traitement_rec.isin(["aucun","autre"])]
df_exp3

In [ ]:
gb_data = df_exp3.groupby(["q5_family_logi", "q5_logiciel_traitement_rec"]).agg(nb=("q45_clé","size")).sort_values(["q5_family_logi","nb"], ascending=[False,False]).reset_index()

In [ ]:
# Initialize the matplotlib figure
fig, ax = plt.subplots(1, 2, figsize=(15,8))

# Plot the total crashes
sns.set_color_codes("pastel")


col = "logiciel_traitement"
sns.barplot(x="total_freq", y=col, data=gb_q5.sort_values("nb", ascending=False),
            label="Non", color="b", ax=ax[0])
sns.barplot(x="nb", y=col, data=gb_q5.sort_values("nb", ascending=False),
            label="Oui", color="r", ax=ax[0])
titre = "5) Quels langages de programmation/logiciels de traitement de données utilisez-vous ?"
ax[0].yaxis.set_label_text("")
ax[0].xaxis.set_label_text("")
ax[0].set_title(titre.replace("<i>","(").replace("</i>",")"))

gb_data2 = gb_data.loc[gb_data.nb>1]
sns.barplot(x="nb", y="q5_logiciel_traitement_rec", data=gb_data2,
            hue="q5_family_logi", ax=ax[1])
titre = "Les logiciels et langages cités par 2 personnes ou plus"
ax[1].yaxis.set_label_text("")
ax[1].xaxis.set_label_text("")
ax[1].set_title(titre.replace("<i>","(").replace("</i>",")"))
    

sns.despine(left=True, bottom=True)
plt.savefig(f"viz/logiciel_traitement_use_by_2people.png", bbox_inches='tight', dpi = 200)

Le graphique ci-dessous donne à voir les logiciels qui sont cités par plus de deux répondants. Il regroupe les questions 5 et 5.1 sur les logiciels et langages utilisés pour le traitement des données. La question 5 proposait 6 outils : Excel, R, python, SAS, Stata et Julia. Pour chacun de ces outils, les personnes interrogées devaient répondre si elles les utilisaient ou non. Avec la question 5.1, elles pouvaient ensuite compléter leur réponse en donnant le nom d'autres outils informatiques. Les 36 personnes qui ont répondu à la question 5.1 ont ainsi cité 26 autres logiciels et langages, ce qui fait au total 48  outils informatiques différents. Pour faciliter la lecture, nous les avons regroupés par "familles". On retrouve ainsi :

* les "logiciels de statistiques" comme SPSS, Jamovi, Stata ;
* les langages de "scripts et de programmations" comme R, Python, Matlab ou C++. Probablement qu'une subdivision de cette catégorie serait justifiée afin de distinguer les langages généralement employés pour l'analyse de données (R, Julia, Matlab, Python) et les langages de programmation comme C++ ;
* les outils de gestion de bibliographie (Zotero, Endnote) ;
* les logiciels de "tableurs" et les feuilles de calculs : Excel, Libre Calc ;
* les langages liés au développement web (css, html, php, javascript) ;
* les outils d'analyse textuelle (Iramuteq, ActiveTigger) ;
* les "CAQDAS" (Computer-Assisted Qualitative Data Analysis Software) comme Nvivo, Qualcoder, Atlas.ti

En termes d'utilisateurs, Excel est l'outil le plus répendu. Deux tiers des répondants (N=79) y recourent pour traiter leurs données. Ce sont ensuite les langage R (N=38) et Python (N=23) qui sont les plus utilisés. Jamovi (N=9) est le plus cité des logiciels d'analyse statistique, comme Nvivo (N=6) pour les logiciels CAQDAS. On note par ailleurs qu'une peronne sur cinq dit n'utiliser aucun outil informatique pour traiter les données.


![](viz/logiciel_traitement_use_by_2people.png)


In [ ]:
df01 = df0.copy()

dict_logiciel_traitement2 = {}
dict_logiciel_fam = {}
for row in df_exp3.q45_clé:
    
    dftmp = df_exp3.loc[df_exp3.q45_clé==row]
    logiciel_traitement2 = [x for x in dftmp.q5_logiciel_traitement_rec.unique()]
    family_logi = [x for x in dftmp.q5_family_logi.unique()]
    dict_logiciel_traitement2[row] = "|".join(logiciel_traitement2)
    dict_logiciel_fam[row] = "|".join(family_logi)


In [ ]:
df0["q5_logiciel_traitement_rec"]= df0.q45_clé.map(dict_logiciel_traitement2.get)
df0["q5_family_logi"] = df0.q45_clé.map(dict_logiciel_fam.get)


# on enregistre les nouvelles variables dans le dataframe original
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-07-24.csv", "w") as file_out:
    df0.to_csv(file_out, sep=",", index= False)


In [ ]:
df_col1 = df_col.iloc[0:-5]
df_col1

In [ ]:
dic_new_var = {'name':'13.1 logiciel de traitment',
               'label':'q5_logiciel_traitement_rec',
               'group':'4_logiciel_traitement',
               'personal_data':False,
               'type':'multiple_choice',
               'opened_question':False,
               'type_panda':df0.q5_logiciel_traitement_rec.dtypes,
               'no_question':5,
               'question':df_col.question.loc[df_col.label=="q5_r"].iloc[0],
               "question_family":df_col.question_family.loc[df_col.label=="q5_r"].iloc[0],
               'comment':"Regroupe les questions 5 et 6 sur les logiciels de traitement"
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col1 = pd.concat([df_col1,new_variable], ignore_index = True)
df_col1

dic_new_var = {'name':'13.2 familles de logiciel',
               'label':'q5_family_logi',
               'group':'4_logiciel_traitement',
               'personal_data':False,
               'type':'multiple_choice',
               'opened_question':False,
               'type_panda':df0.q5_family_logi.dtypes,
               'no_question':5,
               'question':df_col.question.loc[df_col.label=="q5_r"].iloc[0],
               "question_family":df_col.question_family.loc[df_col.label=="q5_r"].iloc[0],
               'comment':"Classe les logiciels par famille"
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col1 = pd.concat([df_col1,new_variable], ignore_index = True)
df_col1


## Développement et dépôt de codes ou de logiciels

In [ ]:
dev_code= grouped_question(df0, column="q6_dev_code", index = "q45_clé")
print(dev_code)

df_exp, sw_depot2 = split_multiple_choices(df0, column="q7_software_depot", index="q45_clé", sep = '|')
print("\n### Ouverture du code \n", sw_depot2.to_markdown())

Parallèlement  à l'utilisation d'outils informatiques pour le traitement des données, près d'un quart des répondants ont déjà été amenés à développer ou faire développer du code ou des logiciels dans le cadre de leurs recherches, dont deux n'ont jamais déposé leur code ou logiciel sur un entrepot dédié.

Si le dépôt de code sur Github et Gitlab restent les solutions les plus usitées, 6 personnes déposent leus codes et logiciels ailleurs : 
* "Sur les dépôts SSC des commandes Stata";
* "sur des repositories personnels";
* "sur Hal";
* "sur Ortolang et Huma-Num",
* "Sur des instances auto-hébergées de Gitlab"
* ou "sur l'INPI".


Sur les 32 personnes ayant répondu "Oui" à la question "êtes-vous amené.e à développer ou faire développer des logiciels/du code spécialisé(s) ?", 31 ont donnée des indications plus ou moins précises sur ce qu'elles développent ou font développer en répondant à la question 16. Nous avons analysé les réponses manuellement. On a ainsi 8 personnes qui disent développer des "logiciels variés", des "algorithmes" ou encore des "plateformes, programmes logiciels, œuvres" sans données d'autres précisions. Les chercheurs qui manipulent du code afin de traiter et d'analyser les données constituent le groupe le plus importants (N=14). On observe enfin un petit groupe de personnes (N=7) construisant des outils liés à la collecte de données d'expérience via des jeux ou le logiciel Psytoolkit. 

| Type de développement                  |   nb |   total |   freq |
|:---------------------------------|-----:|--------:|-------:|
| traitement et analyse de données |   15 |      31 |   48,4 |
| programmes et logiciels variés   |    8 |      31 |   25,8 |
| collecte de données              |    7 |      31 |   22,6 |
| diffusion                        |    2 |      31 |    6,5 |


In [ ]:
rec_code={
    "Mes activités de recherche consist à développer des outils spécialisés pour le traitement de la parole et l'analyse des médias":
    {"what":"traitement du son","group":"traitement et analyse de données", "how":""},
    
    "des apps web": 
    {"what":"applications web","group":"programmes et logiciels variés", "how":""},
    
    "Des logiciels variés, allant du materiel d'expérience à des productions directes à être diffusées.":
    {"what":"outils d'expériences", "group":"collecte de données|programmes et logiciels variés", "how":""},
    
    "Questionnaires et programmes d'expérience sur Psytoolkit":
    {"what":"outils d'expérience","group":"collecte de données", "how":"psychtoolkit"},
    
    "Beaucoup d'outils cf. https://samszo.jardindesconnaissances.fr/HDR/cv/cv.html#sec-item299386":
     {"what":"", "group":"programmes et logiciels variés", "how":""},
    
    "Les Commades Stata":
    {"what":"des commandes stata", "group":"traitement et analyse de données", "how":"Stata"},
    
    "Plugin de moteur temps réel pour la XR.":
    {"what":"plugins pour réalité virtuelle", "group":"programmes et logiciels variés", "how":""},
    
    "Je ne l'ai pas fait moi-même mais à grenoble je travaille avec les ingénieurs de la PUD pour fabriqueur un outil semi automatique de pseudonymisation.":
    {"what":"outils de pseudonymisation", "group": "traitement et analyse de données","how":""},
    
    "Une librairie Julia pour du traitement de réseaux.":
    {"what":"analyse de réseau", "group":"traitement et analyse de données", "how":"Julia"},
    
    "plateformes, programmes logiciels, œuvres":
    {"what":"programmes et logiciels variés", "group":"programmes et logiciels variés", "how":""},
    
    "Logiciels de synthèse et traitement du son":
    {"what":"traitement du son","group":"traitement et analyse de données", "how":""},
    
    "Outil de transcription et d'annotation de données": 
    {"what":"transcription et annotation", "group":"traitement et analyse de données", "how":""},
    
    "https://pablo.rauzy.name/software.html":
    {"what":"", "group":"programmes et logiciels variés","how":""},
    
    "création de site internet pour valorisation et diffusion des activités de recherche, productions des étudiants, etc": 
    {"what":"sites web", "group": "diffusion", "how":""},
    
    "Implémentation d's ou algorithme de démonstration":
    {"what":"implémentation", "group": "programmes et logiciels variés", "how":""},
    
    "jeu sérieux sur tablette": 
    {"what":"jeu sérieux", "group":"collecte de données", "how":""},
    
    "Développez les scripts/méthodologies pour traiter mes données":
    {"what":"scripts", "group":"traitement et analyse de données", "how":""},
    
    "Plateforme PatriMaths, sémathèque : https://sematheque.ahp-numerique.fr":
    {"what":"plateforme","group":"diffusion","how":""},
    
    "Dans le cadre du consortium Projets Time Machine, plusieurs logiciels/codes spécialisés sont des logiciels: d'analyse morphologique (MorphAL = Analyse morphologique) qui a énté intégrée comme un plug-in dans QGis ; Amado-on-line (logiciel de visualisation des matrices graphiques le suivant de Bertin)":
    {"what":"analyse morphologique", "group":"traitement et analyse de données", "how":"qgis"},
    
    "prédication de C02 dans Paris":
    {"what":" prédiction", "group":"traitement et analyse de données", "how":""},
    "Scripts Python. En ce moment : prestation avec le CERES pour met à jour Digipower Academy, en partenariat avec l'association suisse Personaldata.io":
    {"what":"applications web", "group":"programmes et logiciels variés","how":"python"},
    
    "jouer aux jeux à 1, à 2 et joueurs et + selon les jeux (ou problèmes considérés)":
    {"what":"jeux", "group":"collecte de données", "how":""},
    
    "Des logiciels d'Interaction Humain-Machine transportante des appareils pétéraux d'interactions (recueil de données de protocoles d'interaction)":
    {"what":"logiciels d'interaction humain-machine", "group":"collecte de données", "how":""},
    
    "Des cahiers Jupyter pour l'analyse des données de recherche":
    {"what":"notebook", "group":"traitement et analyse de données","how":"jupyter"},
    
    "Nous avons avec l'INA une IA (gpt-oss-120) pour thématiser de gros cormus de vidéos":
    {"what":"classification automatique", "group":"traitement et analyse de données", "how":"IA générative"},
    
    "une série d'outil de modélisation à base d'approche profond lerning":
    {"what":"modélisation",  "group":"traitement et analyse de données", "how":"deep learning"},
    
    "Dans un projet de recherche je participe à participer, nous faisons réaliser un HTR de formulaires manuscrits.":
    {"what":"reconnaissance de caractère", "group":"traitement et analyse de données","how":""},
    
    "des code pour les expériences à l'aide d'outils type PsychToolkit":
    {"what":"outils d'expérience", "group":"collecte de données", "how":"psychtoolkit"},
    "TEI/XML adapté à une correspondance du xviiie siècle":
    {"what":"transcription et annotation", "group":"traitement et analyse de données","how":"TEI/XML"},
    
    "Base de de donation TOFLLIT18 (issu d'un projet ANR)":
    {"what":"base de données", "group":"collecte de données", "how":""},
    
    "des scripts R pour du traitement d'analyse de despageses":
    {"what":"scripts", "group":"traitement et analyse de données", "how":""}
}

In [ ]:
list_exemple_code = [x for x in df0.q6_1_dev_exemple.loc[~df0.q6_1_dev_exemple.isna()]]
new_rec_code = {}
for n, x in enumerate(rec_code):
    new_key = list_exemple_code[n]
    new_rec_code[new_key]= rec_code[x]

new_rec_code


In [ ]:
list_row= []
for x in new_rec_code:

    id_row = df0.q45_clé.loc[df0.q6_1_dev_exemple==x].values

    if len(id_row) > 0:
        exemple = new_rec_code[x]
        dict_row = {"q45_clé":id_row[0],
              "q6_1_dev_what":exemple["what"].lower(),
              "q6_1_dev_group":exemple["group"].lower(),
              "q6_1_dev_how": exemple["how"].lower()}
        list_row.append(dict_row)

dftp = pd.DataFrame.from_dict(list_row)
dftp


In [ ]:


df01 = df0.merge(dftp, on = ["q45_clé"], how="left")



In [ ]:
# on enregistre les nouvelles variables dans le dataframe original
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-07-24.csv", "w") as file_out:
    df01.to_csv(file_out, sep=",", index= False)

In [ ]:
dic_new_var = {'name':'16. dev_exemple',
               'label':'q6_1_dev_what',
               'group':'5_developpement',
               'personal_data':False,
               'type':'simple_nominal',
               'opened_question':False,
               'type_panda':df01.q6_1_dev_what.dtypes,
               'no_question':16,
               'question':df_col.question.loc[df_col.label=="q6_1_dev_exemple"].iloc[0],
               'comment':"Recodage de la question 16. Restitue le type d'objet développé : traitement du son, jeu, base de donnée, site web"
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col1 = pd.concat([df_col1,new_variable], ignore_index = True)
df_col1

In [ ]:
dic_new_var = {'name':'16. dev_exemple',
               'label':'q6_1_dev_group',
               'group':'5_developpement',
               'personal_data':False,
               'type':'simple_nominal',
               'opened_question':False,
               'type_panda':df01.q6_1_dev_group.dtypes,
               'no_question':16,
               'question':df_col.question.loc[df_col.label=="q6_1_dev_exemple"].iloc[0],
               'comment':"Classe les exemples de logiciels développés dans des groupes : outils de traitement et d'analyse, outils de diffusion, collecte de données, etc."
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col1 = pd.concat([df_col1,new_variable], ignore_index = True)
df_col1

In [ ]:
dic_new_var = {'name':'16. dev_exemple',
               'label':'q6_1_dev_how',
               'group':'5_developpement',
               'personal_data':False,
               'type':'simple_nominal',
               'opened_question':False,
               'type_panda':df01.q6_1_dev_how.dtypes,
               'no_question':16,
               'question':df_col.question.loc[df_col.label=="q6_1_dev_exemple"].iloc[0],
               'comment':"Précise le langage ou l'outil de programmation utilisé"
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col1 = pd.concat([df_col1,new_variable], ignore_index = True)
df_col1

In [ ]:
df_col1.to_csv("../le_questionnaire/dico_variable.csv", sep=",", index= False)

In [ ]:
df_exp, gb_data1 = split_multiple_choices(df01.loc[~df01.q6_1_dev_group.isna()], column='q6_1_dev_group', index="q45_clé", sep = '|')
print(gb_data1.drop(columns=["total_freq"]).round(decimals=1).to_markdown(index=False))